# Exercise 3
### Big Bang nucleosynthesis

1-4) In order to calculate the primordial nucleosynthesis, the thermodynamic evolution has to be known. Complete the following script by replacing all instances of \'___\' to create a file containing the thermodynamic properties after the Big Bang:

In [30]:
# First import the basic packages
import numpy as np
import matplotlib.pyplot as plt


# Weak freezeout temperature in MeV
kB_Tweak = 0.8

# Photon to baryon ratio
eta = 6.11e-10

# Create a function for the temperature in dependence of the time
T9 = lambda t: 13.336/(np.sqrt(t))

# Create a function for the density
rho = lambda t: 3.376e4*eta*T9(t)**3

# Calculate the time of weak freezeout.
# Remember that the weak freezeout temperature is given in 
# MeV while T9 is given in GK. The conversion factor between them
# is: 1 MeV -> 11.604519 GK
first_time = 2.064
# Create a time array containing a logarithmic time grid starting from first_time up to 10^5 s.
time = np.logspace(np.log10(first_time), 5, 200)


# Now the only missing ingredient is the electron fraction
# For this define some constants
mn    =  939.5654133 # mass neutron    [ MeV / c**2 ]
mp    =  938.2720813 # mass proton     [ MeV / c**2 ]
Delta = mn - mp      # mass difference [ MeV / c**2 ]

# Followed from Maxwell Boltzmann expression for the chemical potentials
n_p_ratio = np.exp(-Delta/0.8)

# From n+p = 1:
neutron_freezout = n_p_ratio/(1. + n_p_ratio)
proton_freezout  = 1. - neutron_freezout
# The initial electron fraction is therefore given as:
# Same as neutron abundance, because there are only nucleons
ye_freezout = proton_freezout/(neutron_freezout+proton_freezout) 


# Save the trajectory 
out = np.array([time,T9(time),rho(time),[ye_freezout for i in range(len(time))]]).T
np.savetxt('bbn_trajectory.dat',out,header='time[s], T9[GK], density[g/cm^3], Ye \n')

5) Modify the parameter file BigBang.par  in order to calculate the primordial nucleosynthesis with WinNet. For the calculation use the files that are contained in your folder. 
1. net_bbn : List with contained nuclei in the network
2. bbn_winvn : List with properties of the nuclei
3. bbn_reactions : Reaction rates included
4. track_nuclei: Nuclei to be tracked

When the parameter file is ready, run WinNet.

In [31]:
! /home/teaching/WinNet/bin/winnet BigBang.par >OUT 2>ERR

After your calculation plot the abundance of the most abundant nuclei versus time.

3) Create many trajectories for different etas and run all of them with WinNet

In [27]:
# Define an array of etas (consider the interval of etas shown in the plot from the lecture)
etas =np.logspace(-9, -10, 50)

In [28]:
# For every value of eta, create a different trajectory
for eta in etas:
    
    # Modify only the variables depending on eta
    rho = lambda t: 3.376e4*eta*T9(t)**3
    out = [[ti, T9(ti), rho(ti), ye_freezout] for ti in time]
    
    # Save every trajectory with a different name, in a separate folder
    np.savetxt('trajectories/bbn_trajectory'+str(eta)+".dat",out,header='time[s], T9[GK], density[g/cm^3], Ye \n')

In [29]:
# For every value of eta, run the network
for eta in etas:
    
    # Modify the parameter file with the name of the specific trajectory
    
    # Read the parameter file
    with open('BigBang.par','r') as fr:
        lines = fr.readlines()
    
    # Modify the line corresponding to the trajectory name
    lines[22] = 'trajectory_file          = \"trajectories/bbn_trajectory'+str(eta)+'.dat\"\n'
    
    # Write the new parameter file
    with open('BigBang.par','w') as fw:
        fw.writelines(lines)
    
    # Run the network
    !../../WinNet/bin/winnet BigBang.par >OUT 2>ERR
    
    # Store the results with a different name, in a separate folder (in particular we need the final abundances)
    !mv finab.dat results/finab{eta}.dat
    print('Ran network for',eta)

Ran network for 1e-09
Ran network for 9.540954763499925e-10
Ran network for 9.102981779915227e-10
Ran network for 8.68511373751352e-10
Ran network for 8.28642772854686e-10
Ran network for 7.906043210907701e-10
Ran network for 7.543120063354608e-10
Ran network for 7.196856730011529e-10
Ran network for 6.866488450042998e-10
Ran network for 6.551285568595495e-10
Ran network for 6.250551925273976e-10
Ran network for 5.963623316594636e-10
Ran network for 5.689866029018305e-10
Ran network for 5.428675439323859e-10
Ran network for 5.179474679231202e-10
Ran network for 4.941713361323838e-10
Ran network for 4.71486636345739e-10
Ran network for 4.498432668969453e-10
Ran network for 4.2919342601287785e-10
Ran network for 4.0949150623804193e-10
Ran network for 3.9069399370546207e-10
Ran network for 3.727593720314938e-10
Ran network for 3.556480306223136e-10
Ran network for 3.3932217718953296e-10
Ran network for 3.23745754281764e-10
Ran network for 3.088843596477485e-10
Ran network for 2.9470517025

In [35]:
Y(n), Y(p), Y(d), Y(t), Y(he3), Y(he4), Y(li6), Y(li7), Y(be7)= np.loadtxt("tracked_nuclei.dat",unpack=True)

SyntaxError: cannot assign to function call here. Maybe you meant '==' instead of '='? (1880219611.py, line 1)